# 06 — Inference & API
**Resume Classification — Deep Learning Project**

Notebook ini membangun fungsi inference dan API FastAPI yang siap dipakai website.

---
**Input  :** `models/final/`, `models/tokenizer/`, `models/final/label_encoder.pkl`  
**Output :** `api/app.py` — API siap di-deploy

## 1. Setup & Import

In [7]:
import os
import pickle
import torch
import numpy as np
from transformers import AutoModelForSequenceClassification
FINAL_DIR  = '../models/final/'
TOK_DIR    = '../models/tokenizer/'
MAX_LENGTH = 512

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')

Device : cuda


## 2. Load Model, Tokenizer & Label Encoder

In [8]:
# Load label encoder first
with open(os.path.join(FINAL_DIR, 'label_encoder.pkl'), 'rb') as f:
    le = pickle.load(f)

# Load tokenizer
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(TOK_DIR)
print('Tokenizer loaded.')

# Load model
model = AutoModelForSequenceClassification.from_pretrained(FINAL_DIR)
model = model.to(DEVICE)
model.eval()
print('Model loaded.')

# Label pendek untuk response API
LABEL_MAP = {
    'Business Intelligence, Business Object Resumes'   : 'BI / Business Object',
    'Business Analyst (BA) Resumes'                    : 'Business Analyst',
    'Datawarehousing, ETL, Informatica Resumes'        : 'DWH / ETL / Informatica',
    'Java Developers/Architects Resumes'               : 'Java Developer',
    'Network and Systems Administrators Resumes'        : 'Network & SysAdmin',
    'Project Manager Resumes'                          : 'Project Manager',
    'Recruiter Resumes'                                : 'Recruiter',
    'SQL Developers Resumes'                           : 'SQL Developer',
    'Web Developer Resumes'                            : 'Web Developer',
}

print(f'Kelas : {le.classes_.tolist()}')

Tokenizer loaded.


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 8029.30it/s]


Model loaded.
Kelas : ['Business Analyst (BA) Resumes', 'Business Intelligence, Business Object Resumes', 'Datawarehousing, ETL, Informatica Resumes', 'Java Developers/Architects Resumes', 'Network and Systems Administrators Resumes', 'Project Manager Resumes', 'Recruiter Resumes', 'SQL Developers Resumes', 'Web Developer Resumes']


## 3. Fungsi Preprocessing & Predict

In [9]:
import re

def clean_text(text: str) -> str:
    """Preprocessing teks — sama persis dengan notebook 02."""
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = text.encode('ascii', errors='ignore').decode()
    text = re.sub(r"[^a-zA-Z0-9'\s]", ' ', text)
    text = re.sub(r'\b\d+\b', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def predict(text: str, top_k: int = 3) -> dict:
    """
    Prediksi kategori dari teks resume.

    Parameters
    ----------
    text  : teks resume mentah (belum dibersihkan)
    top_k : jumlah prediksi teratas yang dikembalikan

    Returns
    -------
    dict dengan key:
        predicted_category : label pendek kelas terbaik
        confidence         : probabilitas kelas terbaik (0-1)
        top_k              : list {category, confidence} top-k
    """
    # 1. Clean
    clean = clean_text(text)

    # 2. Tokenisasi
    encoding = tokenizer(
        clean,
        max_length     = MAX_LENGTH,
        padding        = 'max_length',
        truncation     = True,
        return_tensors = 'pt',
    )

    input_ids      = encoding['input_ids'].to(DEVICE)
    attention_mask = encoding['attention_mask'].to(DEVICE)

    # 3. Inference
    with torch.no_grad():
        outputs = model(
            input_ids      = input_ids,
            attention_mask = attention_mask,
        )

    probs     = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]
    top_idx   = np.argsort(probs)[::-1][:top_k]

    # 4. Decode label
    best_label_full = le.classes_[top_idx[0]]
    best_label      = LABEL_MAP.get(best_label_full, best_label_full)

    top_k_result = [
        {
            'category'   : LABEL_MAP.get(le.classes_[i], le.classes_[i]),
            'confidence' : round(float(probs[i]), 4),
        }
        for i in top_idx
    ]

    return {
        'predicted_category' : best_label,
        'confidence'         : round(float(probs[top_idx[0]]), 4),
        'top_k'              : top_k_result,
    }


print('Fungsi predict() siap.')

Fungsi predict() siap.


## 4. Test Fungsi Predict

In [ ]:
# Contoh resume palsu untuk test
sample_resume = """
Senior Java Developer with 8 years of experience in designing and developing
enterprise-level applications using Spring Boot, Hibernate, and Microservices.
Strong knowledge of RESTful API design, Docker, and Kubernetes.
Experience with AWS cloud services including EC2, S3, and RDS.
Led a team of 5 developers on multiple projects delivered on time.
"""

result = predict(sample_resume, top_k=3)

print('HASIL PREDIKSI')
print(f"Kategori  : {result['predicted_category']}")
print(f"Confidence: {result['confidence']*100:.2f}%")
print()
print('Top 3 prediksi:')
for i, item in enumerate(result['top_k'], 1):
    print(f"  {i}. {item['category']:25s} {item['confidence']*100:.2f}%")

=== HASIL PREDIKSI ===
Kategori  : Java Developer
Confidence: 68.64%

Top 3 prediksi:
  1. Java Developer            68.64%
  2. Recruiter                 16.55%
  3. Network & SysAdmin        7.44%


In [11]:
# Test dengan beberapa sampel dari test set
import pandas as pd

test_df = pd.read_pickle('../data/processed/test.pkl')
samples = test_df.sample(5, random_state=42)

print('=== TEST 5 SAMPEL DARI TEST SET ===')
print()
for i, row in samples.iterrows():
    result = predict(row['clean_text'])
    actual = LABEL_MAP.get(row['category'], row['category'])
    status = '✓' if result['predicted_category'] == actual else '✗'
    print(f"{status} Aktual   : {actual}")
    print(f"  Prediksi : {result['predicted_category']} ({result['confidence']*100:.1f}%)")
    print()

=== TEST 5 SAMPEL DARI TEST SET ===

✓ Aktual   : Recruiter
  Prediksi : Recruiter (74.6%)

✗ Aktual   : Web Developer
  Prediksi : Java Developer (82.4%)

✗ Aktual   : Network & SysAdmin
  Prediksi : DWH / ETL / Informatica (96.2%)

✗ Aktual   : Web Developer
  Prediksi : DWH / ETL / Informatica (63.2%)

✓ Aktual   : Java Developer
  Prediksi : Java Developer (82.9%)



## 5. Buat File API (FastAPI)

In [ ]:
api_code = '''
"""
Resume Classification API
Jalankan dengan: uvicorn app:app --host 0.0.0.0 --port 8000 --reload
"""

import os
import re
import pickle
import numpy as np
import torch
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ── Konfigurasi ──────────────────────────────────────────────
FINAL_DIR  = '../models/final/'
TOK_DIR    = '../models/tokenizer/'
MAX_LENGTH = 512
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

LABEL_MAP = {
    'Business Intelligence, Business Object Resumes'   : 'BI / Business Object',
    'Business Analyst (BA) Resumes'                    : 'Business Analyst',
    'Datawarehousing, ETL, Informatica Resumes'        : 'DWH / ETL / Informatica',
    'Java Developers/Architects Resumes'               : 'Java Developer',
    'Network and Systems Administrators Resumes'        : 'Network & SysAdmin',
    'Project Manager Resumes'                          : 'Project Manager',
    'Recruiter Resumes'                                : 'Recruiter',
    'SQL Developers Resumes'                           : 'SQL Developer',
    'Web Developer Resumes'                            : 'Web Developer',
}

# ── Load model saat server start (sekali saja) ───────────────
print(f"Loading model ke {DEVICE}...")
tokenizer = AutoTokenizer.from_pretrained(TOK_DIR)
model     = AutoModelForSequenceClassification.from_pretrained(FINAL_DIR)
model     = model.to(DEVICE)
model.eval()

with open(os.path.join(FINAL_DIR, 'label_encoder.pkl'), 'rb') as f:
    le = pickle.load(f)

print("Model siap!")

# ── FastAPI App ───────────────────────────────────────────────
app = FastAPI(
    title       = 'Resume Classification API',
    description = 'Klasifikasi kategori resume menggunakan DistilBERT',
    version     = '1.0.0',
)

# Izinkan request dari website (CORS)
app.add_middleware(
    CORSMiddleware,
    allow_origins     = ["*"],   # ganti dengan domain website kamu di production
    allow_methods     = ["*"],
    allow_headers     = ["*"],
)

# ── Schema Request & Response ─────────────────────────────────
class ResumeRequest(BaseModel):
    text  : str
    top_k : int = 3

class PredictionItem(BaseModel):
    category   : str
    confidence : float

class ResumeResponse(BaseModel):
    predicted_category : str
    confidence         : float
    top_k              : list[PredictionItem]

# ── Helper ───────────────────────────────────────────────────
def clean_text(text: str) -> str:
    text = re.sub(r'http\\S+|www\\.\\S+', ' ', text)
    text = re.sub(r'\\S+@\\S+', ' ', text)
    text = text.encode('ascii', errors='ignore').decode()
    text = re.sub(r"[^a-zA-Z0-9\'\\s]", ' ', text)
    text = re.sub(r'\\b\\d+\\b', ' ', text)
    text = re.sub(r'\\s+', ' ', text).strip()
    return text

# ── Endpoints ────────────────────────────────────────────────
@app.get('/')
def root():
    return {'message': 'Resume Classification API aktif', 'device': str(DEVICE)}


@app.post('/predict', response_model=ResumeResponse)
def predict_resume(request: ResumeRequest):
    if not request.text.strip():
        raise HTTPException(status_code=400, detail='Teks resume tidak boleh kosong')

    if len(request.text) < 50:
        raise HTTPException(status_code=400, detail='Teks resume terlalu pendek (min 50 karakter)')

    clean  = clean_text(request.text)
    enc    = tokenizer(
        clean,
        max_length     = MAX_LENGTH,
        padding        = 'max_length',
        truncation     = True,
        return_tensors = 'pt',
    )

    input_ids      = enc['input_ids'].to(DEVICE)
    attention_mask = enc['attention_mask'].to(DEVICE)

    with torch.no_grad():
        outputs = model(
            input_ids      = input_ids,
            attention_mask = attention_mask,
        )

    probs   = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]
    top_idx = np.argsort(probs)[::-1][:request.top_k]

    best    = le.classes_[top_idx[0]]
    top_res = [
        PredictionItem(
            category   = LABEL_MAP.get(le.classes_[i], le.classes_[i]),
            confidence = round(float(probs[i]), 4),
        )
        for i in top_idx
    ]

    return ResumeResponse(
        predicted_category = LABEL_MAP.get(best, best),
        confidence         = round(float(probs[top_idx[0]]), 4),
        top_k              = top_res,
    )


@app.get('/health')
def health_check():
    return {'status': 'ok', 'model': 'distilbert-base-uncased', 'num_labels': 9}
'''

os.makedirs('../api', exist_ok=True)
with open('../api/app.py', 'w') as f:
    f.write(api_code.strip())

print('File API disimpan ke: api/app.py')

## 6. Cara Menjalankan API

In [ ]:
print("""
=== CARA MENJALANKAN API ===

1. Buka terminal, masuk ke folder api/:
   cd api

2. Jalankan server:
   uvicorn app:app --host 0.0.0.0 --port 8000 --reload

3. API bisa diakses di:
   http://localhost:8000

4. Dokumentasi interaktif (Swagger UI):
   http://localhost:8000/docs

5. Test endpoint /predict dari website:

   fetch('http://localhost:8000/predict', {
     method: 'POST',
     headers: { 'Content-Type': 'application/json' },
     body: JSON.stringify({ text: '<isi resume>', top_k: 3 })
   })
   .then(res => res.json())
   .then(data => console.log(data))

=== CONTOH RESPONSE ===
{
  "predicted_category": "Java Developer",
  "confidence": 0.9512,
  "top_k": [
    { "category": "Java Developer",   "confidence": 0.9512 },
    { "category": "Web Developer",    "confidence": 0.0301 },
    { "category": "SQL Developer",    "confidence": 0.0087 }
  ]
}
""")

## 7. Ringkasan Seluruh Pipeline

In [ ]:
print('=' * 60)
print('PIPELINE SELESAI — RESUME CLASSIFICATION')
print('=' * 60)
print("""
NOTEBOOK
  01_eda.ipynb           ✓ eksplorasi & insight dataset
  02_preprocessing.ipynb ✓ cleaning, encoding, split
  03_tokenisasi.ipynb    ✓ tokenisasi & PyTorch Dataset
  04_training.ipynb      ✓ fine-tuning Longformer
  05_evaluasi.ipynb      ✓ metrik, confusion matrix
  06_inference.ipynb     ✓ predict & API FastAPI

STRUKTUR AKHIR
  resume_project/
  ├── data/raw/          ← dataset asli
  ├── data/processed/    ← output preprocessing & tokenisasi
  ├── models/final/      ← model terbaik siap pakai
  ├── models/checkpoint/ ← semua checkpoint per epoch
  ├── notebooks/         ← semua .ipynb
  └── api/app.py         ← API FastAPI untuk website

CARA PAKAI DI WEBSITE
  POST http://localhost:8000/predict
  Body: { "text": "<isi resume>", "top_k": 3 }
""")
print('=' * 60)

---
**Seluruh pipeline selesai.** Model siap dipakai di website melalui API FastAPI.